# Credit Ratings Monitoring
## Automated Detection and Analysis of Credit Rating Events
This workflow demonstrates the capabilities of Bigdata to monitor specific events. In this example, we track credit ratings news for a Company and extract relevant features to track changes over time.

## Why It Matters
Credit rating changes or outlook revisions can have immediate effects on corporate bond spreads, equity valuations, and counterparty risk assessments. For traders, portfolio managers, and credit analysts, staying ahead of these developments is critical to anticipate market reactions and adjust exposure before the news is fully priced in.

## What It Does
This workflow systematically detects, labels, and summarizes event-related news for a selected watchlist of companies and entities using the Bigdata API for content retrieval and large language models for feature extraction. By customizing the prompts, keywords, and parameters, this framework can be adapted to monitor any type of corporate event or regulatory development - from credit ratings and earnings announcements to regulatory changes and strategic developments. The output includes both structured datasets and analytical reports for monitoring or backtesting.

## How It Works
The workflow implements a four-step agentic pipeline built on Bigdata API:

- **Content Retrieval & Enhancement**: Search for entities and event-specific keywords using the Bigdata Search API. Run queries over configurable time windows with parallel processing, collecting raw content with metadata and enriching results with surrounding context.

- **Feature Extraction & Validation**: Use LLM-powered analysis to identify entity relationships, extract structured event features, and validate classifications. The prompts can be customized to extract any necessary features for different event types and domains.

- **Advanced Analytics Generation**: Derive timestamped analytics with event-specific scoring and sentiment analysis tailored to your monitoring requirements.

- **Report generation**: Produce dated timelines of events with supporting quotes, source links, and exportable datasets for further analysis or integration into existing workflows.

## A Real World Use Case
This cookbook demonstrates the complete workflow through a practical example: tracking credit rating updates and outlook revisions for Tesla over a three-year period. You’ll learn how to transform unstructured rating-related news into structured insights that highlight rater-ratee relationships, analyst commentary, and market implications.

## Setup and Imports

## Async Compatibility Setup

**Run this cell first** - Required for Google Colab, Jupyter Notebooks, and VS Code with Jupyter extension:

### Why is this needed?

Interactive environments (Colab, Jupyter) already have an asyncio event loop running. When `bigdata-research-tools` makes async API calls (like to OpenAI), you'll get this error without nest_asyncio:

```
RuntimeError: asyncio.run() cannot be called from a running event loop
```

The `nest_asyncio.apply()` command patches this to allow nested event loops.

💡 **Tip**: If you're unsure which environment you're in, just run the cell below - it won't hurt in any environment!

In [1]:
import datetime
start = datetime.datetime.now()

In [2]:
try:
    import asyncio
    asyncio.get_running_loop()
    import nest_asyncio; nest_asyncio.apply()
    print("✅ nest_asyncio applied")
except (RuntimeError, ImportError):
    print("✅ nest_asyncio not needed")

✅ nest_asyncio applied


## Environment Setup

The following cell configures the necessary path for the analysis

In [3]:
import os
import sys

current_dir = os.getcwd()
if current_dir not in sys.path:
    sys.path.append(current_dir)
print(f"✅ Local environment setup complete")

✅ Local environment setup complete


## Load Credentials

In [4]:
from dotenv import load_dotenv
from pathlib import Path

script_dir = Path(__file__).parent if '__file__' in globals() else Path.cwd()
load_dotenv(script_dir / '.env')

BIGDATA_API_KEY = os.getenv('BIGDATA_API_KEY')
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

if not all([BIGDATA_API_KEY, OPENAI_API_KEY]):
    print("❌ Missing required environment variables")
    raise ValueError("Missing required environment variables. Check your .env file.")
else:
    print("✅ Credentials loaded from .env file")

✅ Credentials loaded from .env file


## Import Required Libraries

Below is the Python code required for setting up our environment and importing necessary libraries.

In [5]:
import pandas as pd
from bigdata_client import Bigdata
from bigdata_client.models.search import DocumentType
from src.knowledge_graph_manager import *
from src.search_enhanced import search_enhanced
from src.feature_extractor import FeatureExtractor, clean_credit_ratings_features_dataset
from src.summary_generator import SummaryGenerator
from src.visuals import ReportVisualizer

## Optional: Plotly Display Configuration

For better visualization rendering, you can also set the Plotly renderer:

In [6]:
import plotly.io as pio
import plotly.graph_objects as go

try:
    import os
    if 'JUPYTERHUB_SERVICE_PREFIX' in os.environ or 'JPY_SESSION_NAME' in os.environ:
        pio.renderers.default = 'jupyterlab'
        print("✅ Plotly configured for JupyterLab")
    else:
        pio.renderers.default = 'plotly_mimetype+notebook'
        print("✅ Plotly configured for Jupyter/VS Code")
except:
    pio.renderers.default = 'notebook'
    print("✅ Plotly configured with fallback renderer")

interactive_plots = True  # Set to False to generate static plots

✅ Plotly configured for Jupyter/VS Code


## Define Output Paths

We define the output paths for our Trump reelection impact analysis results.

In [7]:
output_dir = "report"
os.makedirs(output_dir, exist_ok=True)

export_path = f"{output_dir}/credit_ratings_monitor.csv"

## Connecting to Bigdata

Create a Bigdata object with your credentials.

In [8]:
bigdata = Bigdata(api_key=BIGDATA_API_KEY)


## Defining Your Event Monitoring (Feature Extraction) Context and Parameters

To perform an event monitoring and feature extraction analysis, we need to define a few key parameters:

- **Company Names** (`company_names`): The set of companies to monitor events for (e.g. your portfolio or watchlist)

- **Rating Agencies Names** (`rating_agencies_names`): (Optional) The list of entities you want to be co-mentioned with your entities and events

- **Keywords** (`keywords`): The keywords characterizing the event

- **Time Period** (`start_date_query` and `end_date_query`): The date range over which to run the search

- **Frequency** (`frequency`): The frequency of the date ranges to search over. Supported values:
    - Y: Yearly intervals.
    - M: Monthly intervals.
    - W: Weekly intervals.
    - D: Daily intervals

- **Document Limit** (`document_limit`): The maximum number of documents to return per query to Bigdata API.

- **Batch Size** (`batch_size`): The number of entities to include in a single batched query.

- **Document Type** (`document_type`): Specify which documents to search over (transcripts, filings, news)

- **Model Selection** (`llm_model`): The AI model used for semantic analysis and topic classification

In [ ]:
# ===== Context Definition ====
company_names = ['Tesla']
rating_agencies_names = ['S&P Global', 'Fitch Ratings Inc', "Moody's Corp"]
keywords = ['credit rating'] #Select the keyword related to the event you need to track, i.e. credit ratings

# ===== Specify Time Range =====
start_date_query = '2021-09-01' # Start date
end_date_query = '2025-01-01' # End date
frequency = 'D'

# ===== Query Configuration =====
document_limit = 50  # Maximum number of retrieved documents for each day
batch_size = 1 #number of companies to process in each batch
document_type = DocumentType.NEWS  # Scope of search

# ===== LLM Specification =====
model = 'openai::gpt-4o-mini'

## Portfolio Selection

Define your watchlist starting from the companies name. For the purpose of this example, the workflow selects **Tesla**. The Entity ID is retrieved by leveraging Bigdata.com’s Knowledge Graph.

In [10]:
companies, full_company_names, company_objects = get_entity_ids(company_names)

In order to boost the accuracy of the retrieval process, the workflow includes a selection of control entities in the queries, that is, entities that have to appear alongside Tesla in the search results. For the purpose of this example, the workflow selects the list of Credit Rating Agencies (CRAs), such as **S&P**, **Moody’s**, and **Fitch**.

In [11]:
rating_agencies, full_agencies_names, rating_agency_companies = get_entity_ids(rating_agencies_names)

## Content Retrieval from Bigdata Search API

The workflow searches news content using the Bigdata API to find articles mentioning Tesla, rating agencies, and credit rating keywords. The search runs across daily windows with parallel processing for efficiency.

The `search_enhanced` function retrieves not only the matching text chunks but also their surrounding context (previous and next paragraphs) to provide richer information for analysis.

Results are stored in the `contextualized_chunks` DataFrame for feature extraction.

In [12]:
contextualized_chunks = search_enhanced(
    companies=companies,
    keywords=keywords,
    sentences=None,
    control_entities=rating_agencies,
    start_date=start_date_query,
    end_date=end_date_query,
    scope=document_type,
    frequency=frequency,
    document_limit=document_limit,
    batch_size=batch_size,
    enhance_search=True,
)

About to run 3 queries
Example Query: And(Entity('DD3BB1'), Entity('CFF97C', '65A2CE', '3461CF'), Keyword('credit rating')) over date range: AbsoluteDateRange('2021-09-01T00:00:00', '2021-12-31T23:59:59')


Querying Bigdata...: 100%|██████████| 3/3 [00:00<00:00,  4.25it/s]


[BigdataEntity(id='767F86', name='Caterpillar Inc.', volume=None, description='Caterpillar Inc. (formerly Caterpillar Tractor Co.) is engaged in the manufacturing of construction and mining equipment. The company was founded in 1925.', entity_type='COMP', company_type='Public', country='United States', sector='Industrials', industry_group='Industrial Engineering', industry='Commercial Vehicles and Trucks', ticker='CAT', webpage='http://www.caterpillar.com', isin_values=['CA14913M1086', 'US1491231015'], cusip_values=['149123101', '14913M108', 'P0R682207'], sedol_values=['0180162', '2180201', '2378277', 'BRF1S02'], listing_values=['XETR:CAT1', 'XFRA:CAT1', 'XLON:0Q18', 'XNYS:CAT'], product_type=None, product_owner=None, organization_type=None, position=None, employer=None, nationality=None, gender=None, place_type=None, region=None, landmark_type=None, entity_type_name=None, concept_level_2=None, concept_level_3=None, concept_level_4=None, concept_level_5=None), BigdataEntity(id='CFF97C'

Processing news results...:   0%|          | 0/10 [00:00<?, ?it/s]

## Features Augmentation

### Entity Role Detection

The LLM agent is prompted to *label* the sentences extracted to identify the role played by the entities detected in each sentence, detecting raters and ratees. A unique identifier for the tuple of entity name, document headline, and augmented text is created and the content is sent to the LLM for role detection.

In [13]:
feature_extractor = FeatureExtractor(llm_model=model)

In [14]:
df_labeled = feature_extractor.assign_entity_roles(contextualized_chunks, 'contextualized_chunk_text', ['entity_name', 'headline'], action_type='detect')

Querying an LLM...:   0%|          | 0/32 [00:00<?, ?it/s]

Querying an LLM...: 100%|██████████| 32/32 [00:03<00:00, 10.35it/s]


### Entity Role Validation

Subsequently, we prompt the LLM agent to *validate* the roles identified in the previous steps. We provide the original text, the label assigned and the motivation, and we instruct the LLM to either confirm or correct the role assigned to each entity.

In [15]:
df_labeled_valid = feature_extractor.assign_entity_roles(df_labeled, 'contextualized_chunk_text', ['entity_name','headline', 'motivation','label'], action_type='validate')

Querying an LLM...:   0%|          | 0/32 [00:00<?, ?it/s]

Querying an LLM...: 100%|██████████| 32/32 [00:02<00:00, 11.78it/s]


### Features Augmentation

The LLM is instructed to augment the features related to long-term and short-term credit ratings, credit outlooks, and analysts’ comments in a multi-step approach leveraging three different prompts. The prompts can be customized to extract any necessary features.

**Prompt 1** is designed to extract the following: 
- **Credit Rating**: Extract the overall credit rating assigned to Tesla.
- **Credit Action**: Extract any change or affirmation of the credit rating assigned to Tesla, categorized as:
    - **Upgrade**: An improvement in the rating.
    - **Downgrade**: A decrease in the rating.
    - **Affirmed**: Rating confirmed with no change.
    - **Corrected**: Adjusted due to an error.
    - **Withdrawn**: The rating is removed.
    - **Reinstated**: A withdrawn rating is restored.
    
- **Credit Status**: Any additional information regarding the credit rating status, categorized as:
    - **Provisional Rating**: A preliminary rating.
    - **Matured or Paid in Full**: When the obligation reaches maturity.
    - **No Rating**: Rating declined or unavailable.
    - **Published**: Officially issued or announced.
    
- **Credit Outlook**: The credit outlook mentioned by the rater, and any related mention of the credit rating assessed in the coming weeks, months or years.
    - **Positive**: Suggests potential improvement.
    - **Negative**: Indicates potential downgrade.
    - **Stable**: No expected change.
    - **Developing**: Change possible based on future events.

- **Credit Watchlist**: Any mention of Tesla being placed in a credit watchlist for review of the credit rating. Labelled as:
    - **Watch**: The rating is on a watchlist.
    - **Watch Positive**: Potential upgrade.
    - **Watch Negative**: Suggests downgrade.
     - **Watch Removed**: No longer active.
     - **Watch Unchanged**: Status remains without change in expectation.

In [16]:
df_labeled_valid['date'] = pd.to_datetime(df_labeled_valid['timestamp_utc']).dt.date

In [17]:
exploded_df = feature_extractor.group_text_and_labels(df_labeled_valid, group_columns=['date', 'sentence_id', 'headline', 'source_name', 'contextualized_chunk_text', 'url'], role_column='validated_label', entity_column='entity_name')

In [18]:
features_extracted = {}

In [19]:
df_credit_ratings = feature_extractor.extract_single_feature(exploded_df, feature_type='credit_ratings', text_col='contextualized_chunk_text', additional_prompt_fields=['rater_entity', 'ratee_entity', 'unclear_entities'])
features_extracted['credit_ratings'] = df_credit_ratings

Querying an LLM...:   0%|          | 0/18 [00:00<?, ?it/s]

Querying an LLM...: 100%|██████████| 18/18 [00:01<00:00,  9.38it/s]


**Prompt 2** is designed to extract the following: 
- **Short Term Credit Rating**: Any credit rating assigned to Tesla and specifically referred to a short-term debt instrument, if mentioned.
- **Long Term Credit Rating**: Any credit rating assigned to Tesla and specifically referred to a long-term debt instrument, if mentioned.
- **Debt Instrument**: The debt instruments under study.

In [20]:
df_debt_instruments = feature_extractor.extract_single_feature(exploded_df, feature_type='debt_instruments', text_col='contextualized_chunk_text', additional_prompt_fields=['rater_entity', 'ratee_entity', 'unclear_entities'])
features_extracted['debt_instruments'] = df_debt_instruments

Querying an LLM...:   0%|          | 0/18 [00:00<?, ?it/s]

Querying an LLM...: 100%|██████████| 18/18 [00:01<00:00,  9.01it/s]


**Prompt 3** is designed to extract the following: 
- **Key Drivers**: Any motivating the credit rating or outlook decision, and influencing the credit quality of the ratee entity, including, but not limited to:
    - Cash flow generation (e.g. earnings, revenues, dividends, assets)
    - Insider trading, stock prices, stock picks
    - Capital structure changes (e.g. equity actions, acquisitions, mergers)
- **Forward Guidance**: Capture any forward guidance discussed regarding current or future credit ratings, including any potential changes or outlook updates.


In [21]:
df_drivers_guidance = feature_extractor.extract_single_feature(exploded_df, feature_type='drivers_guidance', text_col='contextualized_chunk_text', additional_prompt_fields=['rater_entity', 'ratee_entity', 'unclear_entities'])
features_extracted['drivers_guidance'] = df_drivers_guidance

Querying an LLM...:   0%|          | 0/18 [00:00<?, ?it/s]

Querying an LLM...: 100%|██████████| 18/18 [00:03<00:00,  5.12it/s]


In [22]:
df_ext = feature_extractor._combine_features(exploded_df, features_extracted)

In [23]:
df_ext['ratee_entity_rp_entity_id'] = df_ext['ratee_entity'].map(dict(zip([company.name for company in company_objects], [company.id for company in company_objects])))
df_ext = df_ext.sort_values('date').reset_index(drop=True)

## Deriving a Structured Dataframe of Advanced Analytics

The workflow provides a timestamped dataframe of credit ratings news with advanced analytics generated through the feature augmentation process. This dataset can be exported in CSV for Excel for further analysis, such as validation, augmentation, or backtesting.

In [24]:
df_clean = clean_credit_ratings_features_dataset(df_ext)
df_clean.head(10)

,date,sentence_id,headline,source_name,url,contextualized_chunk_text,ratee_entity_rp_entity_id,ratee_entity,rater_entity,credit_rating,credit_outlook,credit_action,credit_status,credit_watchlist,short_term_credit_rating,long_term_credit_rating,debt_instrument,forward_guidance,key_drivers
0,2021-10-22,27780A0CE57AEE659F760AFD200B9758-1,S&P raises Tesla issuer credit rating to 'BB+'...,The Fly,NaN,S&P Global Ratings raised its issuer credit an...,DD3BB1,Tesla Inc.,S&P Global Inc.,BB+,Positive,Upgrade,None,None,None,BB+,None,The outlook reflects the view that Tesla's fre...,"[Positive free operating cash flow generation,..."
1,2021-10-22,8E35C0E1D412DDE5CD1C1943F1E474B4-1,S&P Global Ratings Upgrades Tesla With Positiv...,MT Newswires,NaN,"02:53 PM EDT, 10/22/2021 (MT Newswires) -- S&P...",DD3BB1,Tesla Inc.,S&P Global Inc.,BB+,Positive,Upgrade,None,None,None,BB+,None,S&P Global Ratings upgraded its issuer credit ...,"[solid demand prospects, robust financial metr..."
2,2021-11-09,0F0FFE7AEF0B8468B3285D1738A561D6-13,Seven Elon Musk Tweets That Sent Tesla Shares ...,Bloomberg News,https://www.bnnbloomberg.ca/seven-elon-musk-tw...,4. Arguably Musk's most infamous tweet dropped...,DD3BB1,Tesla Inc.,Moody's Corp.,junk,None,Downgrade,None,None,None,junk,None,None,"[production shortfalls, regulatory scrutiny ov..."
3,2022-01-25,BC75ABAE3459DF8B0892F1C79406E04A-1,Tesla Debt Upgraded By 2 Slabs At Moody's: Why...,Benzinga,NaN,Credit ratings agency Moody's on Tuesday raise...,DD3BB1,Tesla Inc.,Moody's Corp.,Ba1,None,Upgrade,None,None,None,Ba1,None,Moody's raised Tesla Inc's debt rating by two ...,[dominant position in the electric vehicle mar...
4,2022-01-25,FE7FB59DB58B3CBD9BEC38D0D088ACED-1,Tesla Inches Toward Blue-Chip Status With Mood...,Bloomberg News,https://www.bnnbloomberg.ca/tesla-inches-towar...,(Bloomberg) -- Moody's Investors Service Inc.'...,DD3BB1,Tesla Inc.,Moody's Corp.,Cusp of investment grade,None,Upgrade,None,None,None,cusp of investment grade,None,Expectations that Tesla will secure blue-chip ...,"[Current performance of Tesla, Ramping up of c..."
5,2022-05-18,D397C6C7B751126C250AE910D7BB4F9C-1,Why Elon Musk Isn't Bothered About Tesla's Jun...,Benzinga,NaN,"Tesla, Inc.'s (NASDAQ:TSLA) credit rating is b...",DD3BB1,Tesla Inc.,Moody's Corp.,Ba1,None,None,None,None,None,Ba1,None,None,"[financial strength, cash flow from operations..."
6,2022-05-18,D397C6C7B751126C250AE910D7BB4F9C-1,Why Elon Musk Isn't Bothered About Tesla's Jun...,Benzinga,NaN,"Tesla, Inc.'s (NASDAQ:TSLA) credit rating is b...",DD3BB1,Tesla Inc.,S&P Global Inc.,BB+,None,None,None,None,None,BB+,None,None,"[credit rating is below investment grade, fina..."
7,2022-09-03,019CA1157BC710F48AAA05274F9F15F8-1,Elon Musk Calls Out Moody's As 'Irrelevant' In...,Benzinga,NaN,"Tesla, Inc. (NASDAQ:TSLA) investors harbor a s...",DD3BB1,Tesla Inc.,S&P Global Inc.,BB+,None,None,None,None,None,BB+,None,Moody's hinted at moving the company's credit ...,"[current rating of BB+ by S&P, which is one st..."
8,2022-09-03,019CA1157BC710F48AAA05274F9F15F8-1,Elon Musk Calls Out Moody's As 'Irrelevant' In...,Benzinga,NaN,"Tesla, Inc. (NASDAQ:TSLA) investors harbor a s...",DD3BB1,Tesla Inc.,Moody's Corp.,Ba1,Positive,Upgrade,None,None,None,Ba1,None,The agency hinted at moving the company's cred...,"[recent two-notch upgrade in January, current ..."
9,2022-09-03,019CA1157BC710F48AAA05274F9F15F8-2,Elon Musk Calls Out Moody's As 'Irrelevant' In...,Benzinga,NaN,"S&P currently rates Tesla a BB+, which is one ...",DD3BB1,Tesla Inc.,Moody's Corp.,Ba1,Negative,Upgrade,None,None,None,Ba1,None,Moody's hinted at moving the company's credit ...,"[Tesla's reliance on a narrow product lineup, ..."


In [25]:
df_clean.to_csv(export_path)

## Report Generation

In this step, the workflow summarizes the timeline of credit ratings news. Summarization is performed in two steps, removing duplicates and repeated events by generating daily summaries and generating a timeline that highlights new information.

Alongside the timeline of events, the workflow creates a final table which synthetizes the credit rating changes by rating agency, and visualizes it in an interactive plot.

In [26]:
summary_generator = SummaryGenerator(llm_model=model)

In [27]:
reports_dict = summary_generator.generate_report_by_entities(df = df_clean, entity_keys = companies, 
                                                             text_col = 'contextualized_chunk_text',
                                                             fields_for_summary = ['date', 'ratee_entity', 'headline','source_name','url','contextualized_chunk_text'])

Processing... (1/1)


Querying an LLM...:   0%|          | 0/6 [00:00<?, ?it/s]

Querying an LLM...: 100%|██████████| 6/6 [00:05<00:00,  1.15it/s]


Generating Company Report...
Extracting Structured Data Table...
Error creating data table: 'data'


## Save and Display Reports

Reports and tables can be customized and exported as HTML files for further analysis.

In [28]:
visualizer = ReportVisualizer(output_dir)

In [29]:
for object in company_objects:
    visualizer.visualize_report_and_table(reports_dict, object, start_date_query, end_date_query)

No data available to visualize for Tesla Inc..
